
# 🩺 Visual AI With Medical Images (In‑Context Learning)

This notebook shows how a **Vision‑Language Model (VLM)** can improve when we give it **examples inside the prompt**.

We will use **Qwen2‑VL**, a model that understands **images + text**.

⚠️ Educational purpose only — this is **NOT real medical diagnosis**.

---

## Goal

Classify a **colorectal cancer images** as:

- **Normal**
- **Tumour**

We will compare:

| Method | Description |
|------|------|
| 0‑shot | No examples |
| 1‑shot | 1 example |
| 3‑shot | 3 examples |

This technique is called **In‑Context Learning (ICL)**.



## Install Libraries
Run this if using **Colab**.


In [1]:

!pip install -q transformers accelerate pillow



## Load the Vision‑Language Model (Qwen2‑VL)


In [6]:

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image
import requests
from io import BytesIO
import torch

model_name = "Qwen/Qwen2-VL-2B-Instruct"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_name)

print("Model loaded!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded!



## Helper Functions


In [3]:
from PIL import Image
import requests
from io import BytesIO

def load_image(url):
    response = requests.get(url)
    return Image.open(BytesIO(response.content)).convert("RGB")


def ask_model(messages, images):
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=images, return_tensors="pt").to(model.device)

    output = model.generate(**inputs, max_new_tokens=256)
    generated_tokens = output[:, inputs.input_ids.shape[-1]:]
    answer = processor.batch_decode(generated_tokens, skip_special_tokens=True)[0]

    return answer.strip()


# Dataset (Simple Example)

We use a few **public colon cancer images**.

Labels:

- **Normal**
- **Tumour**


In [28]:
# Normal / Healthy examples
normal1 = load_image("https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Normal/NORM-TCGA-EMSMAYCE.png")
normal2 = load_image("https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Normal/NORM-TCGA-ETWAIPMI.png")
normal3 = load_image("https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Normal/NORM-TCGA-FPGLEGLC.png")

# Pneumonia examples
tumour1 = load_image("https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Tumour/TUM-TCGA-DKYLNTIT.png")
tumour2 = load_image("https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Tumour/TUM-TCGA-DMPWNDNH.png")
tumour3 = load_image("https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Tumour/TUM-TCGA-DPNMGITS.png")



test_image_normal = load_image("https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Test%20Images/NORM-TCGA-GRNTRQWQ.png")
test_image_tumour = load_image("https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Test%20Images/TUM-TCGA-FRRLAQCS.png")



# Experiment 1 — 0‑Shot (No Examples)

The model only receives the **test image**.


In [36]:

messages = [
    {
        "role": "user",
        "content": [
            {"type":"image","image":test_image_tumour},
            {"type":"text","text":"""
You are a medical assistant helping a pathologist classify colon tissue images.

First explain briefly why the tissue is normal or cancer.

Then output the final diagnosis in EXACTLY the following format:

Reasoning: <short explanation>

Answer: <one word: Normal or Cancer>
"""}
        ]
    }
]

prediction_0shot = ask_model(messages, [test_image_tumour])
print("0‑shot prediction:", prediction_0shot)

0‑shot prediction: Reasoning: The tissue appears to be normal, as it shows a healthy appearance with no signs of abnormal growth or inflammation.

Answer: Normal


Check with the test images to see if they are accurate.


# Experiment 2 — 1‑Shot (One Example)


The model now receives the **test image** along with one **demonstration example** from each normal and healthy cells.


In [40]:


messages = [
    {
        "role":"user",
        "content":[
            {"type":"text","text":"""You are a medical assistant helping a pathologist classify colon tissue images.
.To help you finding the correct answer, we additionally provide you with example images from other patients together with the classification of the tissue (tissue type). Example:"""},
            {"type":"image","image":normal1},
            {"type":"text","text":"""Reasoning: The glandular structure is regular and organized.
Answer: Normal"""},

            {"type":"text","text":"Example:"},
            {"type":"image","image":tumour1},
            {"type":"text","text":"""Reasoning: Disorganized cells and abnormal nuclear shapes indicate tumour.
Answer: Tumour"""},

            {"type":"text","text":""""Now classify the following image using the SAME format." """},
            {"type":"image","image":test_image_tumour}
        ]
    }
]

prediction_1shot = ask_model(messages, [normal1, tumour1, test_image_tumour])
print("1‑shot prediction:", prediction_1shot)


1‑shot prediction: To classify the image, we need to analyze the tissue structure and compare it with the examples provided. Here are the steps to follow:

1. **Identify the tissue type**: Look at the structure of the tissue and compare it with the examples provided.
2. **Analyze the tissue structure**: Look at the shape, size, and arrangement of the cells and the presence of any abnormal features.
3. **Compare with the examples**: Compare the image with the examples provided to determine the correct classification.

Let's analyze the image:

- **Shape and size**: The cells in the image appear to be irregular in shape and size.
- **Structure**: The cells are arranged in a disorganized manner, with no clear pattern or organization.
- **Abnormal features**: There are no clear signs of glandular structures or organized tissue.

Based on these observations, the tissue in the image does not match any of the examples provided. Therefore, the correct classification for this image is:

**No ti


# Experiment 3 — 3‑Shot (Three Examples)

The model now receives the **test image** along with three **demonstration examples** from each normal and healthy cells.


In [39]:
messages = [
{
"role":"user",
"content":[

{"type":"text","text":
"""You are a medical assistant helping a pathologist classify colon tissue images.

Follow EXACTLY this format:

Reasoning: <short explanation>
Answer: <Normal or Tumour>

Examples:"""},

{"type":"text","text":"Example 1:"},
{"type":"image","image":normal1},
{"type":"text","text":
"""Reasoning: The glandular structure is regular and organized.
Answer: Normal"""},

{"type":"text","text":"Example 2:"},
{"type":"image","image":tumour1},
{"type":"text","text":
"""Reasoning: The tissue shows irregular cell structure and abnormal nuclei.
Answer: Tumour"""},

{"type":"text","text":"Example 3:"},
{"type":"image","image":normal2},
{"type":"text","text":
"""Reasoning: Regular glands and uniform nuclei indicate healthy tissue.
Answer: Normal"""},

{"type":"text","text":"Example 4:"},
{"type":"image","image":tumour2},
{"type":"text","text":
"""Reasoning: Disorganized cells and abnormal nuclear shapes indicate tumour.
Answer: Tumour"""},

{"type":"text","text":"Example 5:"},
{"type":"image","image":tumour3},
{"type":"text","text":
"""Reasoning: The tissue architecture is disrupted and nuclei appear atypical.
Answer: Tumour"""},

{"type":"text","text":"Example 6:"},
{"type":"image","image":normal3},
{"type":"text","text":
"""Reasoning: The glandular pattern is symmetric and normal.
Answer: Normal"""},

{"type":"text","text":"Now classify the following image using the SAME format:"},

{"type":"image","image":test_image_tumour}

]
}
]

prediction_3shot = ask_model(messages, [normal1, tumour1, normal2, tumour2, tumour3, normal3, test_image_tumour])
print("3‑shot prediction:", prediction_3shot)


3‑shot prediction: Reasoning: The tissue shows irregular cell structure and abnormal nuclei.
Answer: Tumour



# Why Performance Improves

More examples help the model:

1. Recognize **visual patterns**
2. Match patterns to **labels**
3. Apply the rule to **new images**

Typical pattern:

**3‑shot > 1‑shot > 0‑shot**


Try to play around with different example images, text prompts, and ordering of the image prompts and see what you observe. You can also try with more shots! Here are some other test image URLs, just change them in the beginning.

Normal:


"https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Test%20Images/NORM-TCGA-GRNTRQWQ.png"

"https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Test%20Images/NORM-TCGA-ICLGGYQK.png"

"https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Test%20Images/NORM-TCGA-SRGRCPWF.png"

"https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Test%20Images/NORM-TCGA-TNNKRWEP.png"

"https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Test%20Images/NORM-TCGA-VLGCQLEG.png"





Tumour:

"https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Test%20Images/TUM-TCGA-FRRLAQCS.png"


"https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Test%20Images/TUM-TCGA-HSMCCWDP.png"

"https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Test%20Images/TUM-TCGA-IPIKKHEE.png"

"https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Test%20Images/TUM-TCGA-KMKVGFNR.png"

"https://raw.githubusercontent.com/fx-erick/in-context-learning-simple-notebook/refs/heads/main/images/Test%20Images/TUM-TCGA-TVVRCVMA.png"

